In [15]:
import duckdb

con = duckdb.connect(
    "taxi_pipeline.duckdb"
)

con.sql("""
SELECT table_schema, table_name
FROM information_schema.tables
""").show()

┌──────────────┬─────────────────────┐
│ table_schema │     table_name      │
│   varchar    │       varchar       │
├──────────────┼─────────────────────┤
│ taxi_data    │ taxi_trips          │
│ taxi_data    │ _dlt_loads          │
│ taxi_data    │ _dlt_pipeline_state │
│ taxi_data    │ _dlt_version        │
└──────────────┴─────────────────────┘



In [18]:
con.sql("""
SELECT 
    MIN(trip_pickup_date_time) AS start_date,
    MAX(trip_pickup_date_time) AS end_date
FROM taxi_data.taxi_trips
""").show()

┌───────────────────────────┬───────────────────────────┐
│        start_date         │         end_date          │
│ timestamp with time zone  │ timestamp with time zone  │
├───────────────────────────┼───────────────────────────┤
│ 2009-06-01 17:03:00+05:30 │ 2009-07-01 05:28:00+05:30 │
└───────────────────────────┴───────────────────────────┘



In [19]:
con.sql("DESCRIBE taxi_data.taxi_trips").show()

┌────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │       column_type        │  null   │   key   │ default │  extra  │
│        varchar         │         varchar          │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ end_lat                │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ end_lon                │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ fare_amt               │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count        │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ payment_type           │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat              │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lon              │ DOUBLE                   │ YES   

In [20]:
con.sql("""
SELECT DISTINCT payment_type
FROM taxi_data.taxi_trips
""").show()

┌──────────────┐
│ payment_type │
│   varchar    │
├──────────────┤
│ Dispute      │
│ CASH         │
│ Credit       │
│ Cash         │
│ No Charge    │
└──────────────┘



In [21]:
con.sql("""
SELECT 
    ROUND(
        100.0 * SUM(CASE WHEN payment_type = 'Credit' THEN 1 ELSE 0 END)
        / COUNT(*),
    2
    ) AS credit_card_percentage
FROM taxi_data.taxi_trips
""").show()

┌────────────────────────┐
│ credit_card_percentage │
│         double         │
├────────────────────────┤
│                  26.66 │
└────────────────────────┘



In [22]:
con.sql("""
SELECT 
    ROUND(SUM(tip_amt), 2) AS total_tips
FROM taxi_data.taxi_trips
""").show()

┌────────────┐
│ total_tips │
│   double   │
├────────────┤
│    6063.41 │
└────────────┘

